# 05 — Dataset do paper e baseline

Este notebook reinicia a replicação fiel de *Interpretability in the Wild* (arXiv:2211.00593) a partir do dataset e das convenções do artigo. Os notebooks didáticos anteriores usavam intervenções simplificadas, como o flip IO/S; daqui em diante, o contraste é o par histórico $p_{IOI}$/$p_{ABC}$, em que os nomes de IO, S1 e S2 são substituídos por três nomes novos.

Objetivos:

1. construir pares determinísticos $p_{IOI}$/$p_{ABC}$ a partir dos 15 templates históricos BABA e das variantes ABBA;
2. verificar nomes single-token e as posições `IO`, `S1`, `S2`, `S1+1` e `END`;
3. reproduzir em $p_{ABC}$ o encadeamento histórico `IO→RAND`, `S→RAND`, `S1→RAND`, com três nomes novos por exemplo;
4. medir diferença de logits IO−S, acurácia IO>S e probabilidade absoluta de IO, incluindo erros-padrão;
5. comparar o lote de desenvolvimento com a baseline geral do artigo — diferença média de logits 3,56, estimada em 100.000 prompts — sem exigir igualdade bit a bit.

O dataset usa `prepend_bos=False` e padding à direita configurado explicitamente. Como o comprimento varia entre exemplos, `END` é reunido por linha e não pode ser substituído por `logits[:, -1]`. O token final, que contém a resposta IO no texto histórico, fica em `END+1`; as métricas leem os logits em `END`, imediatamente antes dessa resposta.

In [1]:
import json
from importlib.metadata import version
from pathlib import Path
import subprocess

import torch

from ioi_replication.paper import (
    build_paired_dataset,
    io_accuracy,
    io_probability,
    logit_difference,
    standard_error,
)
from ioi_replication.runtime import (
    compatibility_mode_config,
    load_gpt2_small,
    select_device,
)

# Estas entradas fixam o lote pareado: tamanho, seed, mistura ABBA/BABA e minibatch.
SEED = 42
N_DEV = 256
BATCH_SIZE = 32
PATTERN = "mixed"
MODEL_NAME = "gpt2"
# Os alvos publicados serão comparados às três leituras calculadas por exemplo.
PAPER_TARGETS = {
    "mean_logit_difference": 3.56,
    "io_accuracy": 0.993,
    "mean_io_probability": 0.49,
}

# A raiz e o dispositivo serão reutilizados ao carregar o modelo e salvar o artefato final.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
device = select_device()
print(f"Dispositivo: {device}; seed do dataset: {SEED}; lote dev: {N_DEV}")

Dispositivo: cuda; seed do dataset: 42; lote dev: 256


## Modelo e pares $p_{IOI}$/$p_{ABC}$

Usamos o GPT-2 Small fornecido pelo TransformerLens 4.3 em modo de compatibilidade. A implementação não importa `transformer_lens.evals.IOIDataset`: ela porta os templates e vocabulários históricos diretamente.

Em cada par, $p_{ABC}$ aplica `IO→RAND`, depois `S→RAND` e finalmente `S1→RAND`. O resultado contém três nomes novos, distintos entre si e dos nomes originais, nos papéis IO, S1 e S2. Template, lugar, objeto, comprimento e posições permanecem alinhados com $p_{IOI}$.

As médias futuras devem usar `groups`, cujas chaves `(pattern, template_id)` separam ABBA de BABA; um lote pequeno não precisa conter todos os 30 grupos possíveis.

In [2]:
# O modelo fornece o tokenizer; dele saem pIOI (prompt original) e pABC (controle com nomes trocados).
model = load_gpt2_small(device)
dataset = build_paired_dataset(
    model.tokenizer,
    n=N_DEV,
    seed=SEED,
    pattern=PATTERN,
)

# Ambos os tensores têm shape [exemplos, tokens] e precisam permanecer alinhados posição a posição.
print("pIOI:", tuple(dataset.ioi_tokens.shape))
print("pABC:", tuple(dataset.abc_tokens.shape))
# `groups` separa os exemplos por padrão e template para agregações sem misturar ABBA e BABA.
print("grupos (pattern, template_id):", len(dataset.groups), "/ até 30")
print("prepend_bos:", dataset.prepend_bos)
assert dataset.prepend_bos is False
assert torch.equal(dataset.ioi_attention_mask, dataset.abc_attention_mask)
assert 1 <= len(dataset.groups) <= 30
assert all(pattern in {"ABBA", "BABA"} for pattern, _ in dataset.groups)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

pIOI: (256, 21)
pABC: (256, 21)
grupos (pattern, template_id): 30 / até 30
prepend_bos: False


In [3]:
rows = torch.arange(len(dataset))
# Cada vetor em `positions` tem shape [exemplos] e aponta o token semântico de cada linha.
pos = dataset.positions

# IO é o objeto indireto; S1/S2 são as duas ocorrências do sujeito e S1+1 é o token seguinte à primeira.
assert torch.equal(dataset.ioi_tokens[rows, pos["IO"]], dataset.io_token_ids)
assert torch.equal(dataset.ioi_tokens[rows, pos["S1"]], dataset.s_token_ids)
assert torch.equal(dataset.ioi_tokens[rows, pos["S2"]], dataset.s_token_ids)
assert torch.equal(pos["S1+1"], pos["S1"] + 1)
# END é a última posição que prevê a resposta; o nome IO esperado aparece em END+1.
assert torch.equal(dataset.ioi_tokens[rows, pos["END"] + 1], dataset.io_token_ids)
assert torch.equal(dataset.abc_tokens[rows, pos["IO"]], dataset.abc_io_token_ids)
assert torch.equal(dataset.abc_tokens[rows, pos["S1"]], dataset.abc_s1_token_ids)
assert torch.equal(dataset.abc_tokens[rows, pos["S2"]], dataset.abc_s2_token_ids)
assert torch.equal(dataset.abc_tokens[rows, pos["END"] + 1], dataset.abc_io_token_ids)
assert torch.equal(pos["END"], dataset.ioi_attention_mask.sum(dim=1) - 2)

# Dois pares concretos tornam visíveis os papéis e preparam a leitura das métricas em END.
for index in range(2):
    m = dataset.metadata[index]
    original = {m.io_name, m.subject_name}
    abc = {m.abc_io_name, m.abc_s1_name, m.abc_s2_name}
    assert len(abc) == 3 and original.isdisjoint(abc)
    print(f"\nExemplo {index} ({m.pattern}, template {m.template_id})")
    print("pIOI:", dataset.ioi_prompts[index])
    print("pABC:", dataset.abc_prompts[index])
    print({name: int(values[index]) for name, values in pos.items()})

print("\nTokenização, papéis pABC e posições semânticas verificados.")


Exemplo 0 (BABA, template 10)
pIOI: After the lunch, Alicia and Matthew went to the restaurant. Alicia gave a kiss to Matthew
pABC: After the lunch, Megan and Thomas went to the restaurant. Eric gave a kiss to Thomas
{'IO': 6, 'S1': 4, 'S2': 12, 'S1+1': 5, 'END': 16}

Exemplo 1 (BABA, template 10)
pIOI: After the lunch, Mark and Andrea went to the school. Mark gave a basketball to Andrea
pABC: After the lunch, Matthew and Ashley went to the school. Robert gave a basketball to Ashley
{'IO': 6, 'S1': 4, 'S2': 12, 'S1+1': 5, 'END': 16}

Tokenização, papéis pABC e posições semânticas verificados.


## Baseline no índice `END`

Os prompts têm comprimentos diferentes e recebem padding. Portanto, `logits[:, -1]` não representa o fim semântico de todos os exemplos. Cada métrica abaixo reúne os logits em `positions["END"]` por exemplo.

Para limitar VRAM, processamos o lote de desenvolvimento em minibatches e descartamos os logits completos após extrair as três métricas.

In [4]:
# Cada lista acumula uma leitura por exemplo antes de qualquer média global.
all_logit_differences = []
all_accuracies = []
all_io_probabilities = []

with torch.inference_mode():
    for start in range(0, len(dataset), BATCH_SIZE):
        stop = min(start + BATCH_SIZE, len(dataset))
        logits = model(dataset.ioi_tokens[start:stop].to(device))
        # Reunimos END por linha e comparamos os tokens IO e S corretos daquele exemplo.
        end = pos["END"][start:stop].to(device)
        io_ids = dataset.io_token_ids[start:stop].to(device)
        s_ids = dataset.s_token_ids[start:stop].to(device)
        all_logit_differences.append(
            logit_difference(
                logits, io_ids, s_ids, end, reduction="none"
            ).cpu()
        )
        all_accuracies.append(
            io_accuracy(
                logits, io_ids, s_ids, end, reduction="none"
            ).cpu()
        )
        all_io_probabilities.append(
            io_probability(logits, io_ids, end, reduction="none").cpu()
        )

# Os vetores concatenados têm shape [N_DEV]; média e erro-padrão alimentam a comparação seguinte.
per_example = {
    "mean_logit_difference": torch.cat(all_logit_differences),
    "io_accuracy": torch.cat(all_accuracies),
    "mean_io_probability": torch.cat(all_io_probabilities),
}
observed = {name: values.mean().item() for name, values in per_example.items()}
standard_errors = {
    name: standard_error(values).item()
    for name, values in per_example.items()
}
assert observed["mean_logit_difference"] > 0
assert 0 <= observed["io_accuracy"] <= 1
assert 0 <= observed["mean_io_probability"] <= 1
{"observed": observed, "standard_error": standard_errors}

{'observed': {'mean_logit_difference': 3.6441750526428223,
  'io_accuracy': 0.98828125,
  'mean_io_probability': 0.5202313661575317},
 'standard_error': {'mean_logit_difference': 0.10858683288097382,
  'io_accuracy': 0.006739238742738962,
  'mean_io_probability': 0.014613560400903225}}

In [5]:
# Mantemos diferença de logits em unidades brutas e exibimos apenas as métricas proporcionais em porcentagem.
labels = {
    "mean_logit_difference": "Diferença média de logits IO−S",
    "io_accuracy": "Acurácia IO>S",
    "mean_io_probability": "Probabilidade média de IO",
}
# Cada linha liga a estimativa do lote, sua incerteza e o alvo publicado; não é uma métrica normalizada.
for metric, target in PAPER_TARGETS.items():
    value = observed[metric]
    error = standard_errors[metric]
    suffix = "%" if metric != "mean_logit_difference" else ""
    scale = 100 if suffix else 1
    print(
        f"{labels[metric]}: observado {value * scale:.3f} ± "
        f"{error * scale:.3f}{suffix} (erro-padrão) | "
        f"paper {target * scale:.3f}{suffix} | "
        f"desvio {(value - target) * scale:+.3f}{suffix}"
    )

Diferença média de logits IO−S: observado 3.644 ± 0.109 (erro-padrão) | paper 3.560 | desvio +0.084
Acurácia IO>S: observado 98.828 ± 0.674% (erro-padrão) | paper 99.300% | desvio -0.472%
Probabilidade média de IO: observado 52.023 ± 1.461% (erro-padrão) | paper 49.000% | desvio +3.023%


## Interpretação e limite amostral

O valor 3,56 é a baseline geral de diferença média de logits do artigo, calculada em **100.000 prompts**; não é o resultado esperado exato para qualquer lote menor. O artigo também reporta 99,3% de acurácia e 49% de probabilidade média de IO. Aqui usamos 256 exemplos para manter a execução rápida e a memória previsível. A seed fixa, a mistura específica de ABBA/BABA e diferenças numéricas da versão do modelo podem produzir desvios legítimos.

Não definimos `torch.manual_seed`: a inferência está em modo de avaliação e a seed relevante é somente a usada por `random.Random` na construção dos prompts. Reportamos erro-padrão amostral para tornar explícita a incerteza do lote menor.

Por isso, a comparação verifica direção, ordem de grandeza e desempenho alto, mas não impõe igualdade aos números publicados. A célula final registra a baseline deste lote e sua proveniência; o notebook 06 usa o mesmo contraste IO−S para diagnosticar a lista de heads publicada.

In [6]:
# Commit, dtype e versões ligam as métricas ao ambiente exato que as produziu.
git_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=PROJECT_ROOT,
    text=True,
).strip()
model_dtype = str(next(model.parameters()).dtype).removeprefix("torch.")

# O artefato reúne entradas, grupos, resultados e desvios para os notebooks seguintes reutilizarem a baseline.
result = {
    "paper": "arXiv:2211.00593",
    "seed": SEED,
    "config": {
        "model_name": MODEL_NAME,
        "dtype": model_dtype,
        "n_examples": N_DEV,
        "paper_n_examples": 100_000,
        "batch_size": BATCH_SIZE,
        "pattern": PATTERN,
        "prepend_bos": dataset.prepend_bos,
        "device": str(device),
        "torch_version": torch.__version__,
        "transformer_lens_version": version("transformer-lens"),
        "transformer_lens_compatibility_mode": compatibility_mode_config(),
        "dataset_sha256": dataset.dataset_sha256,
        "git_commit": git_commit,
        "n_groups": len(dataset.groups),
    },
    "paper_targets": PAPER_TARGETS,
    "observed": observed,
    "standard_error": standard_errors,
    "deviation_from_paper": {
        metric: observed[metric] - target
        for metric, target in PAPER_TARGETS.items()
    },
}

# A gravação não altera as leituras acima; apenas persiste a referência que fecha este notebook.
output_path = PROJECT_ROOT / "results/paper" / f"baseline_seed_{SEED}_n_{N_DEV}.json"
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(
    json.dumps(result, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print("Resultado salvo em", output_path.relative_to(PROJECT_ROOT))
result

Resultado salvo em results\paper\baseline_seed_42_n_256.json


{'paper': 'arXiv:2211.00593',
 'seed': 42,
 'config': {'model_name': 'gpt2',
  'dtype': 'float32',
  'n_examples': 256,
  'paper_n_examples': 100000,
  'batch_size': 32,
  'pattern': 'mixed',
  'prepend_bos': False,
  'device': 'cuda',
  'torch_version': '2.15.0+cu132',
  'transformer_lens_version': '4.3.0',
  'transformer_lens_compatibility_mode': {'disable_warnings': False,
   'no_processing': False,
   'fold_ln': True,
   'center_writing_weights': True,
   'center_unembed': True,
   'fold_value_biases': True,
   'refactor_factored_attn_matrices': False},
  'dataset_sha256': 'cc24fb0ddfc5f820a335d3a59b6e42cffb298fe34f77a9dabfe3a202af21e5c8',
  'git_commit': 'e88048c460f27992ba221fdaceb7052c219e2911',
  'n_groups': 30},
 'paper_targets': {'mean_logit_difference': 3.56,
  'io_accuracy': 0.993,
  'mean_io_probability': 0.49},
 'observed': {'mean_logit_difference': 3.6441750526428223,
  'io_accuracy': 0.98828125,
  'mean_io_probability': 0.5202313661575317},
 'standard_error': {'mean_log